# 02 - Neural networks across three evaluation protocols

Research question: does the choice of evaluation protocol affect reported NIDS
performance more than the choice of neural architecture?

Two factors, three levels each:

| | P1 naive | P2 corrected | P3 zero-day |
|---|---|---|---|
| A1 MLP | | | |
| A2 1D-CNN | | | |
| A3 Autoencoder | | | |

Read rows for the architecture effect, columns for the protocol effect, then
compare the two spreads. That comparison is the answer.

Gold tables come from notebook 01. The protocol is a property of the data, so
all nine cells train identical code against identical interfaces.

## 0. Setup

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             average_precision_score, matthews_corrcoef)

warnings.filterwarnings("ignore")

PROJECT = Path(r"D:\Saad A2")
GOLD    = PROJECT / "lakehouse" / "gold"
RESULTS = PROJECT / "results"
RESULTS.mkdir(exist_ok=True)

BENIGN   = "Benign"
ZERO_DAY = "Bot"
PROTOCOLS = ["p1_naive", "p2_corrected", "p3_zeroday"]

# classes present in all three partitions (written by notebook 01 cell 7.3b)
_ec = json.loads((RESULTS / "eval_classes.json").read_text())
EVAL_CLASSES = _ec["eval_classes"]
print(f"{len(EVAL_CLASSES)} classes scored, {len(_ec['excluded'])} excluded")

# how many seeds per cell. 3 gives a variance estimate, 1 is a quick pass.
N_SEEDS   = 3
MAX_EPOCH = 25
PATIENCE  = 4
BATCH     = 4096   # large batch: far fewer optimiser steps on CPU

torch.set_num_threads(4)
DEVICE = "cpu"
print(f"torch {torch.__version__} on {DEVICE}, {torch.get_num_threads()} threads")

15 classes scored, 0 excluded
torch 2.13.0+cpu on cpu, 4 threads


## 1. Loading and preprocessing

Scaler and imputation values are fitted on train only. Fitting them on the full
dataset would leak test statistics into training - the same class of mistake
this study is about.

In [ ]:
def load_split(protocol, split):
    """Read one Gold partition as float32 features plus string labels."""
    df = pd.read_parquet(GOLD / protocol / split)
    y = df["Label"].to_numpy()
    X = df.drop(columns=["Label"]).astype("float32")
    return X, y


def prepare(protocol):
    """Load all three splits, impute and scale using train statistics only."""
    Xtr, ytr = load_split(protocol, "train")
    Xva, yva = load_split(protocol, "val")
    Xte, yte = load_split(protocol, "test")
    cols = list(Xtr.columns)

    med = Xtr.median()                      # train only
    Xtr, Xva, Xte = (d.fillna(med) for d in (Xtr, Xva, Xte))

    mu, sd = Xtr.mean(), Xtr.std().replace(0, 1.0)      # train only
    Xtr, Xva, Xte = ((d - mu) / sd for d in (Xtr, Xva, Xte))

    out = []
    for d in (Xtr, Xva, Xte):
        a = d.to_numpy(dtype="float32")
        a = np.clip(np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0), -10, 10)
        out.append(a)
    return dict(Xtr=out[0], ytr=ytr, Xva=out[1], yva=yva,
                Xte=out[2], yte=yte, cols=cols)


DATA = {}
for p in PROTOCOLS:
    t0 = time.perf_counter()
    DATA[p] = prepare(p)
    d = DATA[p]
    print(f"{p:14} train {d['Xtr'].shape}  val {d['Xva'].shape}  "
          f"test {d['Xte'].shape}  ({time.perf_counter()-t0:.1f}s)")
print(f"\nP1 has {DATA['p1_naive']['Xtr'].shape[1] - DATA['p2_corrected']['Xtr'].shape[1]} "
      f"extra features - the leaky ones kept for the naive protocol")

p1_naive       train (570003, 79)  val (121860, 79)  test (121785, 79)  (7.0s)
p2_corrected   train (528602, 77)  val (114133, 77)  test (111968, 77)  (5.2s)
p3_zeroday     train (500654, 77)  val (108145, 77)  test (111968, 77)  (5.8s)

P1 has 2 extra features - the leaky ones kept for the naive protocol


In [ ]:
# binary target: benign vs attack. All three architectures can produce this,
# so it is the common ground for comparing the grid. Multi-class macro-F1 is
# reported separately for the two supervised models.
def to_binary(y):
    return (y != BENIGN).astype("int64")

for p in PROTOCOLS:
    d = DATA[p]
    for s in ("tr", "va", "te"):
        d[f"b{s}"] = to_binary(d[f"y{s}"])
    print(f"{p:14} attack rate  train {d['btr'].mean():.3f}  "
          f"val {d['bva'].mean():.3f}  test {d['bte'].mean():.3f}")

p1_naive       attack rate  train 0.508  val 0.509  test 0.509
p2_corrected   attack rate  train 0.470  val 0.473  test 0.466
p3_zeroday     attack rate  train 0.441  val 0.444  test 0.466


## 2. Architectures

Choice is driven by the data, not by what is fashionable:

- records are tabular, 77-79 numeric features, no structure within a record
- each flow is labelled independently, so row order is not a sequence
- heavy class imbalance
- one architecture must be able to flag an attack class it never saw

No LSTM or transformer. Each row is already a time-aggregated summary of a flow,
so feeding consecutive rows to a sequence model would treat file order as time,
which it is not - adjacent rows are unrelated flows between different hosts.

In [ ]:
class MLP(nn.Module):
    """A1. Dense net over the feature vector. The natural fit for tabular data."""
    def __init__(self, n_in, n_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64),   nn.ReLU(), nn.BatchNorm1d(64),  nn.Dropout(0.2),
            nn.Linear(64, n_out))
    def forward(self, x):
        return self.net(x)


class CNN1D(nn.Module):
    """A2. Treats the feature vector as a 1D signal.

    Caveat worth stating in the paper: column order in a flow record is
    arbitrary, and convolution assumes adjacent positions are related. If this
    beats the MLP the gain may be an artefact of column ordering, which a
    column-shuffle ablation would settle.
    """
    def __init__(self, n_in, n_out=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, 5, padding=2),  nn.ReLU(), nn.BatchNorm1d(16), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(32), nn.MaxPool1d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(32 * (n_in // 4), 64), nn.ReLU(),
            nn.Linear(64, n_out))
    def forward(self, x):
        return self.head(self.conv(x.unsqueeze(1)))


class AutoEncoder(nn.Module):
    """A3. Trained on benign traffic only; anomaly score is reconstruction error.

    The only architecture here that can structurally detect an unseen attack
    class, because it never consumes attack labels (Mirsky et al. 2018).
    """
    def __init__(self, n_in):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16))
        self.dec = nn.Sequential(nn.Linear(16, 32), nn.ReLU(),
                                 nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, n_in))
    def forward(self, x):
        return self.dec(self.enc(x))

## 3. Training

Early stopping on validation score. Class weights for the supervised models
because a 5:1 imbalance otherwise pushes everything toward benign.

In [ ]:
def seed_all(s):
    np.random.seed(s); torch.manual_seed(s)


def batches(n, bs, shuffle=True, rng=None):
    # a final batch of 1 row crashes BatchNorm in train mode, so skip it
    idx = np.arange(n)
    if shuffle:
        (rng or np.random).shuffle(idx)
    for i in range(0, n, bs):
        b = idx[i:i + bs]
        if len(b) > 1:
            yield b


def train_supervised(model, Xtr, ytr, Xva, yva, seed, max_epoch=MAX_EPOCH):
    """Train MLP or CNN. Returns the model at its best validation epoch."""
    seed_all(seed)
    rng = np.random.default_rng(seed)
    w = torch.tensor([1.0, (ytr == 0).sum() / max((ytr == 1).sum(), 1)],
                     dtype=torch.float32)
    lossf = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)   # higher lr suits the larger batch

    Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr)
    Xva_t = torch.from_numpy(Xva)

    best, best_state, bad = -1.0, None, 0
    for ep in range(max_epoch):
        model.train()
        for b in batches(len(Xtr), BATCH, rng=rng):
            opt.zero_grad()
            loss = lossf(model(Xtr_t[b]), ytr_t[b])
            loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            pv = model(Xva_t).argmax(1).numpy()
        _, _, f1, _ = precision_recall_fscore_support(
            yva, pv, average="binary", zero_division=0)
        if f1 > best:
            best, best_state, bad = f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    if best_state:
        model.load_state_dict(best_state)
    return model, ep + 1, best


def train_autoencoder(model, Xtr, ytr, Xva, yva, seed, max_epoch=MAX_EPOCH):
    """Train on BENIGN ONLY. Threshold picked on benign validation errors."""
    seed_all(seed)
    rng = np.random.default_rng(seed)
    Xb = Xtr[ytr == 0]                       # benign only - never sees attacks
    Xb_t = torch.from_numpy(Xb)
    lossf = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    best, best_state, bad = np.inf, None, 0
    for ep in range(max_epoch):
        model.train()
        tot = 0.0
        for b in batches(len(Xb), BATCH, rng=rng):
            opt.zero_grad()
            xb = Xb_t[b]
            loss = lossf(model(xb), xb)
            loss.backward(); opt.step()
            tot += loss.item() * len(b)
        ep_loss = tot / len(Xb)
        if ep_loss < best - 1e-5:
            best, best_state, bad = ep_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    if best_state:
        model.load_state_dict(best_state)

    # threshold = 95th percentile of reconstruction error on BENIGN validation
    model.eval()
    with torch.no_grad():
        Xva_b = torch.from_numpy(Xva[yva == 0])
        err = ((model(Xva_b) - Xva_b) ** 2).mean(1).numpy()
    thr = float(np.percentile(err, 95))
    return model, ep + 1, thr


def predict(model, X, kind, thr=None):
    model.eval()
    with torch.no_grad():
        Xt = torch.from_numpy(X)
        if kind == "ae":
            err = ((model(Xt) - Xt) ** 2).mean(1).numpy()
            return (err > thr).astype("int64"), err
        logits = model(Xt)
        prob = torch.softmax(logits, 1)[:, 1].numpy()
        return logits.argmax(1).numpy(), prob

## 4. Metrics

Accuracy is not reported as a headline. With this imbalance a model predicting
benign for everything scores well while detecting nothing - the base rate
fallacy Sommer and Paxson (2010) describe.

False positive rate matters most operationally: at realistic attack base rates
even a small FPR buries an analyst in alerts.

In [ ]:
def evaluate(y_true_bin, y_pred_bin, score, y_true_raw=None):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    out = dict(
        attack_recall=r,                       # fraction of attacks caught
        precision=p,
        f1=f1,
        fpr=fp / max(tn + fp, 1),              # the deployability number
        pr_auc=average_precision_score(y_true_bin, score),
        mcc=matthews_corrcoef(y_true_bin, y_pred_bin),
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

    # per attack family recall, so a good headline cannot hide a class
    if y_true_raw is not None:
        per = {}
        for cls in EVAL_CLASSES:
            if cls == BENIGN:
                continue
            m = y_true_raw == cls
            if m.sum():
                per[cls] = float(y_pred_bin[m].mean())   # recall for that family
        out["per_class_recall"] = per
    return out

## 5. The grid

Nine cells, N_SEEDS runs each. Identical code and identical preprocessing
everywhere - only the Gold table changes.

In [ ]:
ARCHS = {
    "A1_MLP":         ("sup", lambda n: MLP(n)),
    "A2_CNN1D":       ("sup", lambda n: CNN1D(n)),
    "A3_AutoEncoder": ("ae",  lambda n: AutoEncoder(n)),
}

records = []
t_start = time.perf_counter()

for proto in PROTOCOLS:
    d = DATA[proto]
    n_feat = d["Xtr"].shape[1]
    for arch, (kind, build) in ARCHS.items():
        for seed in range(N_SEEDS):
            t0 = time.perf_counter()
            model = build(n_feat)
            if kind == "sup":
                model, epochs, _ = train_supervised(
                    model, d["Xtr"], d["btr"], d["Xva"], d["bva"], seed)
                yp, sc = predict(model, d["Xte"], kind)
            else:
                model, epochs, thr = train_autoencoder(
                    model, d["Xtr"], d["btr"], d["Xva"], d["bva"], seed)
                yp, sc = predict(model, d["Xte"], kind, thr)

            m = evaluate(d["bte"], yp, sc, d["yte"])
            # capture unseen-class recall here rather than retraining later
            bot_mask = d["yte"] == ZERO_DAY
            m["bot_recall"] = float(yp[bot_mask].mean()) if bot_mask.sum() else None
            m.update(protocol=proto, arch=arch, seed=seed,
                     epochs=epochs, secs=round(time.perf_counter() - t0, 1),
                     n_features=n_feat)
            records.append(m)
            print(f"{proto:14} {arch:16} seed {seed}  "
                  f"F1 {m['f1']:.4f}  recall {m['attack_recall']:.4f}  "
                  f"FPR {m['fpr']:.4f}  ({m['secs']}s, {epochs} ep)")

print(f"\ntotal {(time.perf_counter()-t_start)/60:.1f} min")
res = pd.DataFrame(records)
res.to_json(RESULTS / "nb02_runs.json", orient="records", indent=2)

p1_naive       A1_MLP           seed 0  F1 0.9164  recall 0.9044  FPR 0.0718  (74.8s, 8 ep)
p1_naive       A1_MLP           seed 1  F1 0.9236  recall 0.9372  FPR 0.0956  (92.9s, 14 ep)
p1_naive       A1_MLP           seed 2  F1 0.9146  recall 0.9116  FPR 0.0847  (46.7s, 7 ep)
p1_naive       A2_CNN1D         seed 0  F1 0.9369  recall 0.9546  FPR 0.0861  (892.7s, 15 ep)
p1_naive       A2_CNN1D         seed 1  F1 0.9210  recall 0.9212  FPR 0.0821  (537.2s, 9 ep)
p1_naive       A2_CNN1D         seed 2  F1 0.9270  recall 0.9376  FPR 0.0884  (1186.9s, 20 ep)
p1_naive       A3_AutoEncoder   seed 0  F1 0.5801  recall 0.4278  FPR 0.0488  (37.7s, 25 ep)
p1_naive       A3_AutoEncoder   seed 1  F1 0.6302  recall 0.4820  FPR 0.0494  (36.8s, 25 ep)
p1_naive       A3_AutoEncoder   seed 2  F1 0.6141  recall 0.4643  FPR 0.0494  (36.7s, 25 ep)
p2_corrected   A1_MLP           seed 0  F1 0.9005  recall 0.8405  FPR 0.0228  (111.6s, 17 ep)
p2_corrected   A1_MLP           seed 1  F1 0.8974  recall 0.8369  FP

In [ ]:
# mean and spread across seeds
agg = (res.groupby(["protocol", "arch"])
          .agg(f1=("f1", "mean"), f1_sd=("f1", "std"),
               recall=("attack_recall", "mean"), fpr=("fpr", "mean"),
               pr_auc=("pr_auc", "mean"), mcc=("mcc", "mean"))
          .round(4).reset_index())
display(agg)

grid = agg.pivot(index="arch", columns="protocol", values="f1")
grid = grid[PROTOCOLS]
print("\nbinary F1 grid:")
display(grid)

,protocol,arch,f1,f1_sd,recall,fpr,pr_auc,mcc
0,p1_naive,A1_MLP,0.9182,0.0047,0.9178,0.0840,0.9863,0.8339
1,p1_naive,A2_CNN1D,0.9283,0.0080,0.9378,0.0855,0.9876,0.8529
2,p1_naive,A3_AutoEncoder,0.6081,0.0256,0.4580,0.0492,0.8176,0.4676
3,p2_corrected,A1_MLP,0.8987,0.0016,0.8371,0.0225,0.9639,0.8287
4,p2_corrected,A2_CNN1D,0.9046,0.0008,0.8483,0.0237,0.9677,0.8370
5,p2_corrected,A3_AutoEncoder,0.4860,0.0109,0.3402,0.0521,0.7609,0.3690
6,p3_zeroday,A1_MLP,0.8656,0.0004,0.7843,0.0243,0.9484,0.7817
7,p3_zeroday,A2_CNN1D,0.9040,0.0008,0.8507,0.0274,0.9630,0.8348
8,p3_zeroday,A3_AutoEncoder,0.4769,0.0197,0.3323,0.0531,0.7604,0.3599



binary F1 grid:


protocol,p1_naive,p2_corrected,p3_zeroday
arch,,,
A1_MLP,0.9182,0.8987,0.8656
A2_CNN1D,0.9283,0.9046,0.9040
A3_AutoEncoder,0.6081,0.4860,0.4769


In [ ]:
# 5b. Autoencoder re-run to convergence

AE_MAX_EPOCH = 150

ae_records = []
t_start = time.perf_counter()

for proto in PROTOCOLS:
    d = DATA[proto]
    n_feat = d["Xtr"].shape[1]
    for seed in range(N_SEEDS):
        t0 = time.perf_counter()
        model = AutoEncoder(n_feat)
        model, epochs, thr = train_autoencoder(
            model, d["Xtr"], d["btr"], d["Xva"], d["bva"], seed,
            max_epoch=AE_MAX_EPOCH)
        yp, sc = predict(model, d["Xte"], "ae", thr)

        m = evaluate(d["bte"], yp, sc, d["yte"])
        bot_mask = d["yte"] == ZERO_DAY
        m["bot_recall"] = float(yp[bot_mask].mean()) if bot_mask.sum() else None
        m.update(protocol=proto, arch="A3_AutoEncoder", seed=seed,
                 epochs=epochs, secs=round(time.perf_counter() - t0, 1),
                 n_features=n_feat)
        ae_records.append(m)

        converged = "converged" if epochs < AE_MAX_EPOCH else "STILL AT CEILING"
        print(f"{proto:14} seed {seed}  F1 {m['f1']:.4f}  "
              f"recall {m['attack_recall']:.4f}  FPR {m['fpr']:.4f}  "
              f"({epochs} ep, {converged})")

print(f"\ntotal {(time.perf_counter()-t_start)/60:.1f} min")

ae_res = pd.DataFrame(ae_records)
still_capped = (ae_res.epochs >= AE_MAX_EPOCH).sum()
if still_capped:
    print(f"\nWARNING: {still_capped}/{len(ae_res)} runs still hit the ceiling.")
    print("Raise AE_MAX_EPOCH again, or report the limitation explicitly.")
else:
    print("\nall autoencoder runs converged before the ceiling")

# replace the truncated autoencoder rows in the main results
res = pd.concat([res[res.arch != "A3_AutoEncoder"], ae_res], ignore_index=True)
res.to_json(RESULTS / "nb02_runs.json", orient="records", indent=2)

# rebuild the aggregate and grid so later cells use the converged numbers
agg = (res.groupby(["protocol", "arch"])
          .agg(f1=("f1", "mean"), f1_sd=("f1", "std"),
               recall=("attack_recall", "mean"), fpr=("fpr", "mean"),
               pr_auc=("pr_auc", "mean"), mcc=("mcc", "mean"))
          .round(4).reset_index())
grid = agg.pivot(index="arch", columns="protocol", values="f1")[PROTOCOLS]

print("\nupdated grid:")
display(grid.round(4))


p1_naive       seed 0  F1 0.6537  recall 0.5089  FPR 0.0496  (150 ep, STILL AT CEILING)
p1_naive       seed 1  F1 0.7286  recall 0.6005  FPR 0.0497  (150 ep, STILL AT CEILING)
p1_naive       seed 2  F1 0.7244  recall 0.5948  FPR 0.0491  (150 ep, STILL AT CEILING)
p2_corrected   seed 0  F1 0.6409  recall 0.4990  FPR 0.0508  (150 ep, STILL AT CEILING)
p2_corrected   seed 1  F1 0.6235  recall 0.4788  FPR 0.0499  (150 ep, STILL AT CEILING)
p2_corrected   seed 2  F1 0.5193  recall 0.3702  FPR 0.0484  (150 ep, STILL AT CEILING)
p3_zeroday     seed 0  F1 0.5156  recall 0.3669  FPR 0.0492  (150 ep, STILL AT CEILING)
p3_zeroday     seed 1  F1 0.6293  recall 0.4849  FPR 0.0490  (150 ep, STILL AT CEILING)
p3_zeroday     seed 2  F1 0.6286  recall 0.4852  FPR 0.0513  (150 ep, STILL AT CEILING)

total 33.6 min

Raise AE_MAX_EPOCH again, or report the limitation explicitly.

updated grid:


protocol,p1_naive,p2_corrected,p3_zeroday
arch,,,
A1_MLP,0.9182,0.8987,0.8656
A2_CNN1D,0.9283,0.9046,0.9040
A3_AutoEncoder,0.7022,0.5946,0.5911
